# Data Cleaning Notebook

Within this notebook, a core snRNA-seq retina dataset is split between neuronal and non-neuronal cell types and downsampled. The resulting datasets are stored in the same directory as the original. 

* Data must be downsampled if it contains more than 500,000 cells. If downsampled, the resulting datasets are under `data_{structure}_ds.h5ad`. 
* Data must be filtered if it contains unknown or diseased cells. If filtered, the resulting datasets are under `data_{structure}_filter.h5ad`.
* If stratified, the resulting datasets are under `data_{structure}_{strata}.h5ad`.

In this manner, the tag hierarchy goes `data_{structure}_ds_filter_{strata}.h5ad`. 

**Note:** The input data file should be a single-cell RNA-seq dataset in h5ad format, with cell type annotations in the obs dataframe and gene annotations in the var dataframe.

**Note:** If using GitHub for version control and repository sharing, ensure that you add the path to your data folder to the repository's `.gitignore` file, to prevent yourself from exceeding the GitHub's storage limits.

In [1]:
# libraries
import sys
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import celltypist as ct

/opt/miniconda3/envs/nsforest/lib/python3.11/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [2]:
# configs
code_folder = "/Users/vbecker/NSForest-ncRNA" # path to the NSForest-ncRNA folder
sys.path.insert(0, os.path.abspath(code_folder))

data_folder = "../beckersv_data/" # path to folder containing the input data file (.h5ad format)

to_downsample = True # True if you want to downsample the dataset to a specific number of cells, 
                     # False otherwise

to_downsample_ideal = 500000 # ideal number of cells within each strata's final .h5ad file.

seed = 0 # random seed for reproducibility

## Functions

In [18]:
def find_threshold(og_anndata, cluster_header, ideal_total):
    """
    Args:
        og_anndata (adata): Anndata object containing the original dataset
        cluster_header (str): Header of the column in og_anndata.obs that contains the cluster labels
        ideal_total (int): Ideal total number of cells to downsample to

    Returns:
        int: The threshold value to use for downsampling the dataset to the ideal total number of cells
    """
    
    # get counts dataframe
    og_counts = pd.DataFrame(og_anndata.obs[cluster_header].value_counts()).reset_index()
    print("Original Counts:\n" + str(og_counts))
    
    # if there's less cells than the ideal total, just return the number of cells
    if og_counts['count'].sum() <= ideal_total:
        return ideal_total

    # set upper and lower limits for binary search
    upper_lim = ideal_total
    lower_lim = int(ideal_total / len(og_counts.index))
    
    # recurse
    return find_threshold_recursive(og_counts, ideal_total, upper_lim, lower_lim)
    
def find_threshold_recursive(count_data, ideal_total, upper_lim, lower_lim):
    """
    Args:
        count_data (pd.DataFrame): Dataframe containing the counts of each cluster
        ideal_total (int): Ideal total number of cells to downsample to
        upper_lim (int): Upper limit for the binary search
        lower_lim (_type_): Lower limit for the binary search

    Returns:
        int: The threshold value to use for downsampling the dataset to the ideal total number of cells
    """
    
    # get middle lim
    mid_thresh = (upper_lim + lower_lim) // 2
    
    # get new total
    new_counts = count_data['count'].clip(upper=mid_thresh)
    new_total = new_counts.sum()
    
    # base case check
    if abs(new_total - ideal_total) <= ideal_total * 0.02:
        print("Ideal threshold found!")
        print(f"Ideal Threshold : {mid_thresh}, New Total : {new_total}, Ideal Total : {ideal_total}.")
        print("New Counts:\n" + str(new_counts))
        return mid_thresh
    
    # binary search exhausted :[
    if upper_lim - lower_lim <= 1:
        print("Binary search exhausted :[")
        print(f"Ideal Threshold : {mid_thresh}, New Total : {new_total}, Ideal Total : {ideal_total}.")
        print("New Counts:\n" + str(new_counts))
        return mid_thresh

    # sending the recursive case
    if new_total > ideal_total:
        return find_threshold_recursive(
            count_data,
            ideal_total,
            mid_thresh,
            lower_lim
        )
    else:
        return find_threshold_recursive(
            count_data,
            ideal_total,
            upper_lim,
            mid_thresh
        )
    

In [5]:
def downsample(adata, cluster_header, ideal_total, seed, filepath, filename, return_index=False, write_to_file=True):
    """
    Args:
        adata (ad.AnnData): Anndata object containing the dataset to downsample
        cluster_header (str): Header of the column in adata.obs that contains the cluster labels
        ideal_total (int): Ideal total number of cells to downsample to
        seed (int): Random seed for reproducibility
        filepath (str): Path to the folder where the downsampled anndata object will be saved
        filename (str): Name of the file where the downsampled anndata object will be saved
        return_index (bool, optional): Whether to return the indices of the downsampled cells. Defaults to False.
        write_to_file (bool, optional): Whether to write the downsampled anndata object to a .h5ad file. Defaults to True.
        
    Returns:
        np.ndarray: Indices of the downsampled cells if return_index is True, otherwise None
    """
    
    if not (return_index or write_to_file):
        raise ValueError("At least one of return_index or write_to_file must be True.") 
    
    # find threshold
    print(f"Finding threshold for downsampling to {ideal_total} cells...")
    thresh = find_threshold(adata, cluster_header, ideal_total)
    
    # downsample for all cell types
    print(f"Downsampling to maximum {thresh} cells per cluster...")
    idx = ct.samples.downsample_adata(
        adata,
        mode="each",
        by=cluster_header,
        n_cells=thresh,
        random_state=seed,
        return_index=True,
    )
    
    # write to file if requested
    if write_to_file:
        print(f"Writing downsampled anndata object to {filepath + filename}...")
        adata[idx, :].to_memory().write_h5ad(filename = filepath + filename)
        
    # return index if requested
    if return_index:
        print(f"Returning indices of downsampled cells...")
        return idx

## Cleaning Template

In [ ]:
# load the dataset
file = "*.h5ad"
adata_raw = sc.read_h5ad(data_folder + file, backed = "r")
adata_raw

In [ ]:
# sample exploration
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

In [ ]:
# gene exploration 
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

In [ ]:
# gene counts by cell type
print(adata_raw.obs["*"].value_counts()) 

In [ ]:
# gene counts by feature type
print(adata_raw.var["*"].value_counts())

In [ ]:
# defining cluster header
cluster_header = "*" # column name in adata.obs that contains the cluster labels used in NS-Forest

In [ ]:
# check cell counts by cluster
pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index()

In [ ]:
# fix an issue with original data that prevented writing
adata_raw._raw = None 

In [ ]:
# if applicable, filter out unknown/unhealthy and downsample the dataset to a specific number of cells

unknown_col = "unknown"          # column name in adata_raw.obs that contains the unknown labels
unknown_values = ["unknown"]     # values in the unknown_col that indicate unknown cells
unhealthy_col = "unhealthy"      # column name in adata_raw.obs that contains the unhealthy labels
unhealthy_values = ["unhealthy"] # values in the unhealthy_col that indicate unhealthy cells

# compute mask on the adata_raw object
bad_mask = (
    adata_raw.obs[unknown_col].isin(unknown_values)
    | adata_raw.obs[unhealthy_col].isin(unhealthy_values)
)
filter_mask = ~bad_mask
filter_idx = np.flatnonzero(filter_mask)

# temporary backed view
adata_masked = adata_raw[filter_mask, :]

# downsample for neuron cell types
idx = downsample(adata_masked, cluster_header, to_downsample_ideal, seed, data_folder, "data_retina_neuron_ds.h5ad", return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = filter_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = data_folder + "data_retina_ds_neuron.h5ad")

In [ ]:
# if not applicable, just downsample by cell types
downsample(adata_raw, cluster_header, to_downsample_ideal, seed, data_folder, "data_retina_all_ds.h5ad", return_index=False, write_to_file=True)

## Motor Cortex (TBD)

* **Original Dataset Size:** 181.6k cells

* **Number of Cell Types:** 128

## Middle Temporal Gyrus (TBD)

* **Original Dataset Size:** 15.9k cells

* **Number of Cell Types:** 75

## Retina (Downsampled/Stratified)

* **Original Dataset Size:** 3.2M cells

* **Number of Cell Types:** 123

* Stratifying by `majorclass` into neuronal and non-neuronal cell types.

In [6]:
# retina-specific configs
majorclass_values = {
    "neuron": ["AC", # amacrine cell
               "BC", # bipolar cell
               "Cone", 
               "HC", # horizontal cell
               "RGC", # retinal ganglion cell
               "Rod"],
    
    "non-neuron": ["Astrocyte",
                   "MG", # Müller glia
                   "Microglia",
                   "RPE"] # retinal pigment epithelium
}

In [7]:
# load the retina dataset
file = "data_retina_sn.h5ad"
adata_raw = sc.read_h5ad(data_folder + file, backed = "r")
adata_raw

AnnData object with n_obs × n_vars = 3177310 × 35475 backed at '../beckersv_data/data_retina_sn.h5ad'
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_enriched_cell_types', 'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'reported_diseases', 'sex_ontology_term_id', 'majorclass', 'AC_subclass', '

In [8]:
adata_raw.obs_names # sample names
adata_raw.obs.columns # sample metadata

Index(['reference_genome', 'gene_annotation_version', 'alignment_software',
       'intronic_reads_counted', 'donor_id', 'donor_age',
       'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death',
       'donor_living_at_sample_collection', 'sample_id',
       'sample_preservation_method', 'tissue_ontology_term_id',
       'development_stage_ontology_term_id', 'sample_collection_method',
       'tissue_source', 'tissue_type', 'suspension_derivation_process',
       'suspension_dissociation_reagent', 'suspension_enriched_cell_types',
       'suspension_enrichment_factors', 'suspension_uuid', 'suspension_type',
       'tissue_handling_interval', 'library_id', 'assay_ontology_term_id',
       'sequenced_fragment', 'institute', 'library_id_repository',
       'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id',
       'author_cell_type', 'disease_ontology_term_id', 'reported_diseases',
       'sex_ontology_term_id', 'majorclass', 'AC_subclass', 'AC_cluster',


In [9]:
print(adata_raw.var_names) # gene names
print(adata_raw.var.columns) # gene metadata

Index(['ENSG00000243485', 'ENSG00000237613', 'ENSG00000186092',
       'ENSG00000239945', 'ENSG00000239906', 'ENSG00000241860',
       'ENSG00000241599', 'ENSG00000286448', 'ENSG00000236601',
       'ENSG00000284733',
       ...
       'ENSG00000275249', 'ENSG00000274792', 'ENSG00000274175',
       'ENSG00000275869', 'ENSG00000273554', 'ENSG00000277836',
       'ENSG00000278633', 'ENSG00000276017', 'ENSG00000278817',
       'ENSG00000277196'],
      dtype='object', length=35475)
Index(['feature_is_filtered', 'feature_name', 'feature_reference',
       'feature_biotype', 'feature_length', 'feature_type'],
      dtype='object')


In [10]:
print(adata_raw.obs["author_cell_type"].value_counts()) # gene counts by cell type

author_cell_type
Rod       1066056
MG         221612
MG_OFF     200402
MG_ON      151685
FMB        144086
           ...   
HAC90          79
HAC91          74
HAC92          65
HAC93          55
HAC95          39
Name: count, Length: 123, dtype: int64


In [11]:
print(adata_raw.var["feature_type"].value_counts()) # gene counts by feature type

feature_type
protein_coding                        19266
lncRNA                                15491
IG_V_pseudogene                         187
IG_V_gene                               146
TR_V_gene                               106
TR_J_gene                                79
IG_D_gene                                37
TR_V_pseudogene                          33
transcribed_unprocessed_pseudogene       29
transcribed_unitary_pseudogene           19
IG_J_gene                                18
artifact                                 17
IG_C_gene                                14
IG_C_pseudogene                           9
TR_C_gene                                 6
TR_J_pseudogene                           4
TR_D_gene                                 4
processed_pseudogene                      3
IG_J_pseudogene                           3
transcribed_processed_pseudogene          2
TEC                                       1
unprocessed_pseudogene                    1
Name: count, dtype:

In [12]:
cluster_header = "author_cell_type" # column name in adata.obs that contains the cluster labels
                                    # used in NS-Forest
                                    
strata_header = "majorclass" # column name in adata.obs that contains the strata labels

In [13]:
# check cell counts by cluster and strata
print(pd.DataFrame(adata_raw.obs[cluster_header].value_counts()).reset_index())
print(pd.DataFrame(adata_raw.obs[strata_header].value_counts()).reset_index())

    author_cell_type    count
0                Rod  1066056
1                 MG   221612
2             MG_OFF   200402
3              MG_ON   151685
4                FMB   144086
..               ...      ...
118            HAC90       79
119            HAC91       74
120            HAC92       65
121            HAC93       55
122            HAC95       39

[123 rows x 2 columns]
  majorclass    count
0        Rod  1066056
1         BC   691008
2         AC   571579
3        RGC   399605
4         MG   221612
5       Cone   127060
6         HC    80548
7  Astrocyte    14085
8  Microglia     4894
9        RPE      863


In [14]:
# compute neuron mask on the adata_raw object
neuron_mask = adata_raw.obs[strata_header].isin(majorclass_values["neuron"])
neuron_idx = np.flatnonzero(neuron_mask)

# temporary backed view
adata_neuron = adata_raw[neuron_mask, :]

In [15]:
# compute non-neuron mask on the adata_raw object
nonneuron_mask = adata_raw.obs[strata_header].isin(majorclass_values["non-neuron"])
nonneuron_idx = np.flatnonzero(nonneuron_mask)

# temporary backed view
adata_nonneuron = adata_raw[nonneuron_mask, :]

In [16]:
adata_raw._raw = None # fix an issue with original data that prevented writing

In [19]:
# downsample for all cell types
downsample(adata_raw, cluster_header, to_downsample_ideal, seed, data_folder, "data_retina_all_ds.h5ad", return_index=False, write_to_file=True)

Finding threshold for downsampling to 500000 cells...
Original Counts:
    author_cell_type    count
0                Rod  1066056
1                 MG   221612
2             MG_OFF   200402
3              MG_ON   151685
4                FMB   144086
..               ...      ...
118            HAC90       79
119            HAC91       74
120            HAC92       65
121            HAC93       55
122            HAC95       39

[123 rows x 2 columns]
Ideal threshold found!
Ideal Threshold : 7454, New Total : 509041, Ideal Total : 500000.
New Counts:
0      7454
1      7454
2      7454
3      7454
4      7454
       ... 
118      79
119      74
120      65
121      55
122      39
Name: count, Length: 123, dtype: int64
Downsampling to maximum 7454 cells per cluster...
Writing downsampled anndata object to ../beckersv_data/data_retina_all_ds.h5ad...


In [20]:
# downsample for neuron cell types
idx = downsample(adata_neuron, cluster_header, to_downsample_ideal, seed, data_folder, "data_retina_neuron_ds.h5ad", return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = neuron_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = data_folder + "data_retina_ds_neuron.h5ad")

Finding threshold for downsampling to 500000 cells...
Original Counts:
    author_cell_type    count
0                Rod  1066056
1             MG_OFF   200402
2              MG_ON   151685
3                FMB   144086
4            ML_Cone   118559
..               ...      ...
114            HAC90       79
115            HAC91       74
116            HAC92       65
117            HAC93       55
118            HAC95       39

[119 rows x 2 columns]
Ideal threshold found!
Ideal Threshold : 7589, New Total : 494316, Ideal Total : 500000.
New Counts:
0      7589
1      7589
2      7589
3      7589
4      7589
       ... 
114      79
115      74
116      65
117      55
118      39
Name: count, Length: 119, dtype: int64
Downsampling to maximum 7589 cells per cluster...
Returning indices of downsampled cells...


In [21]:
# downsample for non-neuron cell types
idx = downsample(adata_nonneuron, cluster_header, to_downsample_ideal, seed, data_folder, "data_retina_nonneuron_ds.h5ad", return_index=True, write_to_file=False)

# convert indices relative to the neuron view -> original adata
orig_idx = nonneuron_idx[idx]

# index into the original backed object
adata_raw[orig_idx, :].to_memory().write_h5ad(filename = data_folder + "data_retina_ds_nonneuron.h5ad")

Finding threshold for downsampling to 500000 cells...
Original Counts:
  author_cell_type   count
0               MG  221612
1        Astrocyte   14085
2        Microglia    4894
3              RPE     863
Downsampling to maximum 500000 cells per cluster...
Returning indices of downsampled cells...


## Spinal Cord (Filter)

* **Original Dataset Size:** 67.7k cells

* **Number of Cell Types:** 43

* Filtering out `disease` where values are `amyotrophic lateral sclerosis`.

## Breast (Downsample/Filter)

* **Original Dataset Size:** 803.2k cells

* **Number of Cell Types:** 44

* Filtering out `level2` where values are `stripped_nuclei` and `Doublet`.

## Heart (Downsample/Filter)

* **Original Dataset Size:** 704.2k cells

* **Number of Cell Types:** 70

* Filtering out `cell_state` where values are `unclassified`.

## Kidney (Filter)

* **Original Dataset Size:** 304.6k cells

* **Number of Cell Types:** 75

* Filtering out `disease` where values are `acute kidney injury` and `chronic kidney disease`.

## Lung (Downsample)

* **Original Dataset Size:** 584.9k cells

* **Number of Cell Types:** 61